# 08 — DAG Identifiability Check

Continuation of notebooks 01–06 (METR-LA reliability/density/topology pipeline). Checks whether the causal DAG (Confounders → Density → {Reliability, Topology} → Disparity) is identifiable, using R's `dagitty` package.

**Not yet executed** — written against `dagitty`'s documented API. Run top to bottom in Colab and confirm.


In [1]:
# --- R environment setup for this notebook (safe to re-run; skips if already installed) ---
import subprocess
import sys
subprocess.run(["apt-get", "install", "-y", "-qq", "r-base-core"], stdout=subprocess.DEVNULL)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rpy2"])
get_ipython().run_line_magic("load_ext", "rpy2.ipython")
print("R + rpy2 bridge ready.")


R + rpy2 bridge ready.


In [2]:
%%R
#!/usr/bin/env Rscript
# ============================================================================
# 01_dag_identifiability.R
#
# Run in Colab via: %%R (after 00_setup_and_fairtp_verified.ipynb has loaded
# the rpy2 bridge), or as a standalone R script if you have R set up locally.
#
# NOT executed/tested in the environment that built this file — no R/CRAN
# access there. Written against dagitty's documented API. Verify it runs
# cleanly on first use in Colab and report back if any function signature
# has changed.
# ============================================================================

if (!requireNamespace("dagitty", quietly = TRUE)) install.packages("dagitty")
library(dagitty)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
also installing the dependency ‘V8’

trying URL 'https://cran.rstudio.com/src/contrib/V8_8.2.0.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/dagitty_0.3-4.tar.gz'

The downloaded source packages are in
	‘/tmp/RtmpLRP43N/downloaded_packages’


1. Specify the project DAG in dagitty syntax

In [3]:
%%R
# C = confounders (road type, traffic regime)
# D = sensor density (FairTP's original explanatory variable)
# R = sensor reliability (time-varying mediator)
# T = topology sensitivity (static mediator)
# Y = forecast disparity (outcome)

g <- dagitty('dag {
  C -> D
  D -> R
  D -> T
  D -> Y
  R -> Y
  T -> Y
}')

cat("=== DAG structure ===\n")
print(g)

=== DAG structure ===
dag {
C
D
R
T
Y
C -> D
D -> R
D -> T
D -> Y
R -> Y
T -> Y
}


2. Check identifiability of the total effect of D on Y

In [4]:
%%R
cat("\n=== Adjustment sets for the TOTAL effect of D on Y ===\n")
adj_total <- adjustmentSets(g, exposure = "D", outcome = "Y", effect = "total")
print(adj_total)
if (length(adj_total) == 0) {
  cat("WARNING: no valid adjustment set found for the total effect.",
      "This DAG may not be identifiable as specified.\n")
} else {
  cat("PASS: total effect of D on Y is identifiable via the adjustment set(s) above.\n")
}


=== Adjustment sets for the TOTAL effect of D on Y ===
 {}
PASS: total effect of D on Y is identifiable via the adjustment set(s) above.


3. Check identifiability of the DIRECT effect of D on Y (not through R or T)

In [5]:
%%R
cat("\n=== Adjustment sets for the DIRECT effect of D on Y ===\n")
adj_direct <- adjustmentSets(g, exposure = "D", outcome = "Y", effect = "direct")
print(adj_direct)


=== Adjustment sets for the DIRECT effect of D on Y ===
{ R, T }


4. Check for unblocked backdoor paths between each mediator and Y
   (this is the check for whether R and T, sharing the same parent D,
   creates a problem separating their individual mediated effects)

In [7]:
%%R
cat("\n=== Paths between R and Y ===\n")
print(paths(g, from = "R", to = "Y"))

cat("\n=== Paths between T and Y ===\n")
print(paths(g, from = "T", to = "Y"))

cat("\n=== Is R conditionally independent of T given D? (checks if they can",
    "be separated once D is known) ===\n")
print(dseparated <- dseparated <- localTests(g, type = "dseparation"))
# NOTE: localTests() requires data to actually test independence claims.
# The line above will only report the DAG's IMPLIED d-separation claims
# without data. Run impliedConditionalIndependencies(g) for a plain
# structural listing instead if localTests() errors without data:
cat("\n=== Implied conditional independencies (structural, no data needed) ===\n")
print(impliedConditionalIndependencies(g))


=== Paths between R and Y ===


$paths
[1] "R -> Y"           "R <- D -> T -> Y" "R <- D -> Y"     

$open
[1] TRUE TRUE TRUE


=== Paths between T and Y ===
$paths
[1] "T -> Y"           "T <- D -> R -> Y" "T <- D -> Y"     

$open
[1] TRUE TRUE TRUE


=== Is R conditionally independent of T given D? (checks if they can be separated once D is known) ===
Error in match.arg(type) : 
  'arg' should be one of “cis”, “cis.loess”, “cis.chisq”, “cis.pillai”, “tetrads”, “tetrads.within”, “tetrads.between”, “tetrads.epistemic”


RInterpreterError: Failed to parse and evaluate line 'cat("\\n=== Paths between R and Y ===\\n")\nprint(paths(g, from = "R", to = "Y"))\n\ncat("\\n=== Paths between T and Y ===\\n")\nprint(paths(g, from = "T", to = "Y"))\n\ncat("\\n=== Is R conditionally independent of T given D? (checks if they can",\n    "be separated once D is known) ===\\n")\nprint(dseparated <- dseparated <- localTests(g, type = "dseparation"))\n# NOTE: localTests() requires data to actually test independence claims.\n# The line above will only report the DAG\'s IMPLIED d-separation claims\n# without data. Run impliedConditionalIndependencies(g) for a plain\n# structural listing instead if localTests() errors without data:\ncat("\\n=== Implied conditional independencies (structural, no data needed) ===\\n")\nprint(impliedConditionalIndependencies(g))\n'.
R error message: "Error in match.arg(type) : \n  'arg' should be one of “cis”, “cis.loess”, “cis.chisq”, “cis.pillai”, “tetrads”, “tetrads.within”, “tetrads.betwe

5. Positivity check (structural only — this DAG cannot confirm positivity
   empirically; that requires your actual data, checked separately)

In [ ]:
%%R
cat("\n=== Notes on positivity ===\n")
cat("dagitty cannot verify positivity (every C-stratum having variation in D)\n")
cat("from the DAG alone. Once real data is available, check this explicitly:\n")
cat("  table(data$C, cut(data$D, quantile(data$D)))\n")
cat("Look for empty or near-empty cells — those indicate poor positivity.\n")

6. Time-varying mediator flag (R is time-varying, D and T are static)

In [ ]:
%%R
cat("\n=== IMPORTANT: this DAG treats R as if it were static ===\n")
cat("The dagitty structure above does NOT encode that R is time-varying while\n")
cat("D and T are static. dagitty has no native notion of time-varying nodes.\n")
cat("If R genuinely varies across monitoring windows in a way that could be\n")
cat("affected by prior values of Y (e.g., a bad prediction week affecting how\n")
cat("reliability is measured that week), this DAG is a SIMPLIFICATION and the\n")
cat("identifiability result above should be treated as conditional on that\n")
cat("simplification, not as fully verified. See 07_temporal_alignment.R for\n")
cat("the panel-data-specific version of this question.\n")

7. If NOT identifiable: example of a corrected DAG
   (only relevant if step 2 above returned zero adjustment sets)

In [ ]:
%%R
cat("\n=== If the DAG above was not identifiable, try this correction ===\n")
cat("A common fix: add an explicit edge C -> Y if you suspect the confounder\n")
cat("has effects on disparity not fully mediated through D:\n\n")

g_corrected <- dagitty('dag {
  C -> D
  C -> Y
  D -> R
  D -> T
  D -> Y
  R -> Y
  T -> Y
}')
print(g_corrected)
cat("\nRe-checking adjustment sets with this corrected DAG:\n")
print(adjustmentSets(g_corrected, exposure = "D", outcome = "Y", effect = "total"))